# Batch Status Checker

Use this notebook to monitor an Anthropic Message Batch submitted by `classify_benchmark.py`.

**Workflow:**
1. Run `classify_benchmark.py` — it prints the batch job ID.
2. Paste that ID into the `BATCH_ID` cell below.
3. Run cells top-to-bottom. The poll loop (Cell 4) will keep checking until the batch finishes.
4. When done, Cell 5 previews the classifications. Cell 6 writes them to `base_classifications.csv`.

In [14]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import json
import sys
import time
import csv
from datetime import datetime
from pathlib import Path

import anthropic
from dotenv import load_dotenv

# Resolve package root from this notebook's location (notebooks/ → multilingual-bfcl/)
NOTEBOOK_DIR = Path().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT = PACKAGE_ROOT / "data" / "benchmarks"

load_dotenv(PACKAGE_ROOT / ".env")

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env

print(f"Package root : {PACKAGE_ROOT}")
print(f"Anthropic SDK: {anthropic.__version__}")
print("Client ready.")

Package root : C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl
Anthropic SDK: 0.107.0
Client ready.


In [15]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Paste the batch ID printed by classify_benchmark.py here.
# It looks like: msgbatch_01AbCdEfGhIjKlMnOpQrStUv
BATCH_ID = "msgbatch_01QqRKHh5DpqWV1E9nSSS8qm"

# The benchmark category that was classified (used to locate the output CSV).
CATEGORY = "multiple"   # e.g. "multiple", "simple_python", ...

# Poll interval in seconds when running the auto-poll loop (Cell 4).
POLL_INTERVAL_SECONDS = 60

print(f"Batch ID : {BATCH_ID}")
print(f"Category : {CATEGORY}")
print(f"Output   : {DATA_ROOT / CATEGORY / 'base_classifications.csv'}")

Batch ID : msgbatch_01QqRKHh5DpqWV1E9nSSS8qm
Category : multiple
Output   : C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl\data\benchmarks\multiple\base_classifications.csv


In [20]:
# ── Cell 3: Check current status (one-shot) ───────────────────────────────────
def fetch_status(batch_id: str):
    """Retrieve batch metadata and print a formatted status summary."""
    batch = client.messages.batches.retrieve(batch_id)
    rc = batch.request_counts

    total = rc.processing + rc.succeeded + rc.errored + rc.expired + rc.canceled
    done  = rc.succeeded + rc.errored + rc.expired + rc.canceled
    pct   = (done / total * 100) if total else 0

    print(f"Batch ID          : {batch_id}")
    print(f"Processing status : {batch.processing_status}")
    print(f"Created at        : {batch.created_at}")
    if hasattr(batch, 'ended_at') and batch.ended_at:
        print(f"Ended at          : {batch.ended_at}")
    print()
    print(f"  Total requests   : {total}")
    print(f"  Still processing : {rc.processing}")
    print(f"  Succeeded        : {rc.succeeded}")
    print(f"  Errored          : {rc.errored}")
    print(f"  Expired          : {rc.expired}")
    print(f"  Canceled         : {rc.canceled}")
    print(f"  Progress         : {done}/{total}  ({pct:.1f}%)")
    return batch

batch = fetch_status(BATCH_ID)

Batch ID          : msgbatch_01QqRKHh5DpqWV1E9nSSS8qm
Processing status : ended
Created at        : 2026-06-14 14:01:10.417749+00:00
Ended at          : 2026-06-14 14:06:33.452205+00:00

  Total requests   : 132
  Still processing : 0
  Succeeded        : 132
  Errored          : 0
  Expired          : 0
  Canceled         : 0
  Progress         : 132/132  (100.0%)


In [4]:
# ── Cell 4: Auto-poll until the batch ends ────────────────────────────────────
# Run this cell and leave it running. It checks every POLL_INTERVAL_SECONDS.
# Interrupt the kernel (■ button) at any time — it's safe to re-run.

print(f"Polling every {POLL_INTERVAL_SECONDS}s  (interrupt kernel to stop)")
print("-" * 60)

while True:
    batch = client.messages.batches.retrieve(BATCH_ID)
    rc    = batch.request_counts
    total = rc.processing + rc.succeeded + rc.errored + rc.expired + rc.canceled
    done  = rc.succeeded + rc.errored + rc.expired + rc.canceled
    pct   = (done / total * 100) if total else 0
    ts    = datetime.now().strftime("%H:%M:%S")

    status_str = batch.processing_status
    print(f"[{ts}]  {status_str:<12}  {done}/{total} ({pct:.1f}%)  "
          f"ok={rc.succeeded}  err={rc.errored}  exp={rc.expired}")

    if status_str == "ended":
        print()
        print("✓ Batch has ended. Run Cell 5 to retrieve results.")
        break

    time.sleep(POLL_INTERVAL_SECONDS)

Polling every 60s  (interrupt kernel to stop)
------------------------------------------------------------
[16:18:17]  in_progress   0/152 (0.0%)  ok=0  err=0  exp=0


KeyboardInterrupt: 

In [12]:
# ── Cell 5: Retrieve results & preview ───────────────────────────────────────
# Only run this after Cell 4 reports "ended" (or after checking Cell 3 manually).

def parse_classification(raw: str, custom_id: str):
    """Parse the JSON the model returned. Returns None on failure."""
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}") + 1
        if start != -1 and end > start:
            try:
                obj = json.loads(text[start:end])
            except json.JSONDecodeError:
                print(f"[WARN] parse error for {custom_id}: {raw!r}", file=sys.stderr)
                return None
        else:
            print(f"[WARN] no JSON in response for {custom_id}", file=sys.stderr)
            return None

    result = {
        "id":                    custom_id,
        "parameter_type":        str(obj.get("parameter_type",        "")).lower().strip(),
        "convertible_units":     str(obj.get("convertible_units",     "")).lower().strip(),
        "localizable_query":     str(obj.get("localizable_query",     "")).lower().strip(),
        "localizable_parameters":str(obj.get("localizable_parameters","")).lower().strip(),
    }

    # Hard rule: universal → localizable_parameters must be false
    if result["parameter_type"] == "universal" and result["localizable_parameters"] != "false":
        print(f"[FIX] {custom_id}: universal → forcing localizable_parameters=false")
        result["localizable_parameters"] = "false"

    return result


print("Retrieving results from Anthropic...")
rows   = []
errors = []

for item in client.messages.batches.results(BATCH_ID):
    cid = item.custom_id

    if item.result.type == "succeeded":
        content_blocks = item.result.message.content
        raw = next(
            (b.text for b in content_blocks if hasattr(b, "text")),
            ""
        )
        parsed = parse_classification(raw, cid)
        if parsed:
            rows.append(parsed)
        else:
            errors.append({"id": cid, "error": "parse_error", "message": raw[:300]})

    elif item.result.type == "errored":
        err_kind = getattr(item.result.error, "type", "unknown")
        err_msg  = getattr(item.result.error, "message", "")
        errors.append({"id": cid, "error": err_kind, "message": err_msg})
        print(f"[ERR] {cid}: {err_kind} — {err_msg}")

    else:
        # expired or canceled
        errors.append({"id": cid, "error": item.result.type, "message": ""})
        print(f"[{item.result.type.upper()}] {cid}")

print(f"\nRetrieved {len(rows)} successful results, {len(errors)} failures.")

# Preview first 10 rows
if rows:
    print("\nFirst 10 rows:")
    header = ["id", "parameter_type", "convertible_units", "localizable_query", "localizable_parameters"]
    col_w  = [max(len(h), max((len(str(r[h])) for r in rows[:10]), default=0)) for h in header]
    sep    = "  ".join("-" * w for w in col_w)
    fmt    = "  ".join(f"{{:<{w}}}" for w in col_w)
    print(fmt.format(*header))
    print(sep)
    for r in rows[:10]:
        print(fmt.format(*[r[h] for h in header]))

if errors:
    print(f"\nFailed items ({len(errors)}):")
    for e in errors[:10]:
        msg = f"  — {e['message']}" if e.get("message") else ""
        print(f"  [{e['error']}] {e['id']}{msg}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")

Retrieving results from Anthropic...
[ERR] 41: error — 
[ERR] 63: error — 
[ERR] 6: error — 
[ERR] 120: error — 
[ERR] 123: error — 
[ERR] 117: error — 
[ERR] 142: error — 
[ERR] 176: error — 
[ERR] 37: error — 
[ERR] 11: error — 
[ERR] 175: error — 
[ERR] 151: error — 
[ERR] 108: error — 
[ERR] 110: error — 
[ERR] 179: error — 
[ERR] 178: error — 
[ERR] 116: error — 
[ERR] 145: error — 
[ERR] 36: error — 
[ERR] 153: error — 
[ERR] 195: error — 
[ERR] 183: error — 
[ERR] 114: error — 
[ERR] 118: error — 
[ERR] 69: error — 
[ERR] 93: error — 
[ERR] 81: error — 
[ERR] 20: error — 
[ERR] 190: error — 
[ERR] 59: error — 
[ERR] 131: error — 
[ERR] 57: error — 
[ERR] 45: error — 
[ERR] 51: error — 
[ERR] 173: error — 
[ERR] 22: error — 
[ERR] 72: error — 
[ERR] 188: error — 
[ERR] 192: error — 
[ERR] 106: error — 
[ERR] 169: error — 
[ERR] 184: error — 
[ERR] 60: error — 
[ERR] 172: error — 
[ERR] 21: error — 
[ERR] 156: error — 
[ERR] 9: error — 
[ERR] 165: error — 
[ERR] 27: error — 
[ERR]

In [ ]:
# ── Cell 6: Write / merge results into base_classifications.csv ───────────────
# Successful rows from Cell 5 are written to the category's output CSV.
# If the file already exists, existing rows are preserved and new rows are added
# (or overwrite entries with the same ID if OVERWRITE_EXISTING = True).

OVERWRITE_EXISTING = True   # True: update existing IDs; False: skip duplicates

output_path = DATA_ROOT / CATEGORY / "base_classifications.csv"
CSV_COLUMNS = ["id", "parameter_type", "convertible_units", "localizable_query", "localizable_parameters"]

# Load existing rows (if any)
existing: dict[str, dict] = {}
if output_path.exists():
    with open(output_path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            existing[r["id"]] = r
    print(f"Loaded {len(existing)} existing rows from {output_path}")
else:
    print(f"No existing file at {output_path} — creating fresh.")

# Merge
added = updated = skipped = 0
for row in rows:
    eid = row["id"]
    if eid not in existing:
        existing[eid] = row
        added += 1
    elif OVERWRITE_EXISTING:
        existing[eid] = row
        updated += 1
    else:
        skipped += 1

# Write
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(existing.values())

print(f"\nWrote {len(existing)} rows to {output_path}")
print(f"  Added   : {added}")
print(f"  Updated : {updated}")
print(f"  Skipped : {skipped}")
if errors:
    print(f"  Failed items NOT written: {len(errors)}  (see Cell 5 output above)")